# Seção 6 (recursos do `Field`) — pegadinhas, efeitos colaterais e caveats

Complemento da seção: o quadro abaixo reúne as **notas de revisão** já registradas nos notebooks desta pasta, e em seguida vêm
pegadinhas **novas**, cada uma com demonstração executável e a forma de evitar. Teoria da seção: [`00_teoria.md`](00_teoria.md).

## Já registradas nesta seção

| notebook | registro |
|---|---|
| [`04_default_factories`](04_default_factories.ipynb) | o Pydantic **copia** o default mutável (não o troca por uma factory) |
| [`08_projeto_solucao_comentada`](08_projeto_solucao_comentada.ipynb) | fronteiras exatas testadas (1979-12-31 × 1980-01-01; 0 a 6 portas) |

## Novas

**1. `max_length` conta caracteres, não bytes.**

In [1]:
from decimal import Decimal
from pydantic import BaseModel, Field, ValidationError


class City(BaseModel):
    name: str = Field(max_length=4)


for text in ["ação", "👨‍👩‍👧"]:
    try:
        City(name=text)
        verdict = "aceito"
    except ValidationError:
        verdict = "recusado"
    print(f"'{text}': {verdict}; caracteres={len(text)}, bytes UTF-8={len(text.encode())}")

'ação': aceito; caracteres=4, bytes UTF-8=6
'👨‍👩‍👧': recusado; caracteres=5, bytes UTF-8=18


`"ação"` cabe em `max_length=4`, mas ocupa 6 bytes — um `VARCHAR(4)` em bytes (ou um protocolo com limite em bytes) estoura depois
da validação. E o "emoji de família" visível como **um** símbolo conta 5 caracteres. Alinhe a unidade da restrição com a do destino.

**2. `decimal_places` confere o valor normalizado e guarda a representação original.**

In [2]:
class Money(BaseModel):
    amount: Decimal = Field(decimal_places=2)


for raw in ["1.50", "1.500", "1.5E+1"]:
    value = Money(amount=raw).amount
    print(f"{raw!r}: guardado {value!r}, dump {Money(amount=raw).model_dump_json()}")

'1.50': guardado Decimal('1.50'), dump {"amount":"1.50"}
'1.500': guardado Decimal('1.500'), dump {"amount":"1.500"}
'1.5E+1': guardado Decimal('15'), dump {"amount":"15"}


`"1.500"` passa em `decimal_places=2` (zeros à direita não contam) e é guardado e devolvido como `1.500`. Se o destino exige
exatamente duas casas (um arquivo de remessa bancária), quantize explicitamente: `value.quantize(Decimal("0.01"))`.

**3. `repr=False` não é `exclude=True`.**

In [3]:
class Account(BaseModel):
    user: str
    password: str = Field(repr=False)


account = Account(user="ada", password="s3cr3t")
print(f"repr(account): {account!r}")
print(f"account.model_dump(): {account.model_dump()}")

repr(account): Account(user='ada')
account.model_dump(): {'user': 'ada', 'password': 's3cr3t'}


`repr=False` esconde o campo do `repr` (logs, depuração), mas ele continua em todo dump — inclusive na resposta da API. Segredo:
`SecretStr` (mascara no dump) e/ou `exclude=True` (some do dump).

## Veja também

- [`pydantic/02_pydantic_intermediario`](../../02_pydantic_intermediario.ipynb) (caveats): validador não roda no default
  (`validate_default`).
- [`dataclass_factory/01_formatos_de_configuracao`](../../../dataclass_factory/01_formatos_de_configuracao.ipynb) (caveats): `float` para dinheiro.